# Dynamic EV Charging Forecasting using City Traffic Data

This notebook explains the data and model behind the Flask project. It demonstrates exploratory data analysis (EDA), feature preparation, Random Forest regression, evaluation, and optional use of public APIs.

## Data labels and limits

- The CSV and its charging-demand target are synthetic, generated locally for a reproducible class project. They are not measured city traffic or charging sessions.
- Public weather calls in the web app use Open-Meteo. Its historical archive is weather reanalysis, not a direct station observation; its forecast endpoint returns up to 16 days when available.
- Optional TomTom current traffic is the flow for a nearest road segment to the city reference point, not citywide traffic volume. It requires an API key.
- No actual EV counts, charging sessions, citywide traffic counts, or historical traffic feed is configured. The app labels these unavailable and does not fill them with made-up observations.

Sources: [Open-Meteo forecast](https://open-meteo.com/en/docs) · [Open-Meteo historical archive](https://open-meteo.com/en/docs/historical-weather-api) · [TomTom Flow Segment Data](https://docs.tomtom.com/traffic-api/documentation/tomtom-maps/v1/traffic-flow/flow-segment-data).

## 1. Imports and project paths

In [1]:
from pathlib import Path
import json
import sys

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from IPython.display import display
from joblib import dump
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.model_selection import train_test_split

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "data" / "ev_charging_data.csv").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
sys.path.insert(0, str(PROJECT_ROOT))

from config import DATA_PATH, IMPORTANCE_PATH, METRICS_PATH, MODEL_PATH, TRAINING_INSIGHTS_PATH
from model_utils import (
    CATEGORICAL_COLUMNS, FEATURE_COLUMNS, NUMERIC_COLUMNS, TARGET_COLUMN,
    add_engineered_features, build_model, traffic_status_from_speed,
)
from train_model import calculate_feature_importance, calculate_training_insights, save_visualizations

sns.set_theme(style="whitegrid", palette="deep")
print("Project folder:", PROJECT_ROOT)

Project folder: C:\Users\daksh\Documents\Codex\2026-10-03\files-pasted-by-the-user-i\outputs\EV_Charging_Forecasting


## 2. Load the synthetic dataset

A row represents one generated city-hour. Extra fields such as traffic volume and EV count are retained to illustrate EDA and the target generator; the deployed prediction model does not use them.

In [2]:
df = pd.read_csv(DATA_PATH)
print("Shape:", df.shape)
display(df.head())

Shape: (8640, 13)


,date,city,hour,day_of_week,traffic_volume,average_speed,traffic_status,temperature,rainfall,relative_humidity,weather_condition,ev_count,charging_demand
0,2025-01-01,Bengaluru,0,2,266,57.1,free_flow,18.2,4.3,87.0,Rainy,16,8
1,2025-01-01,Bengaluru,1,2,183,66.2,free_flow,21.2,0.0,59.5,Cloudy,15,6
2,2025-01-01,Bengaluru,2,2,388,57.4,free_flow,19.8,0.0,50.4,Clear,16,11
3,2025-01-01,Bengaluru,3,2,308,61.4,free_flow,21.7,0.0,47.1,Clear,17,11
4,2025-01-01,Bengaluru,4,2,321,63.4,free_flow,20.9,0.0,51.9,Clear,15,8


## 3. Data quality and summaries

In [3]:
print("Column types:")
display(df.dtypes.to_frame("dtype"))
print("Missing values:")
display(df.isna().sum().to_frame("count"))
print("Exact duplicate rows:", int(df.duplicated().sum()))
display(df.describe(include="all").T)

Column types:


,dtype
date,str
city,str
hour,int64
day_of_week,int64
traffic_volume,int64
average_speed,float64
traffic_status,str
temperature,float64
rainfall,float64
relative_humidity,float64


Missing values:


,count
date,0
city,0
hour,0
day_of_week,0
traffic_volume,0
average_speed,0
traffic_status,0
temperature,0
rainfall,0
relative_humidity,0


Exact duplicate rows: 0


,count,unique,top,freq,mean,std,min,25%,50%,75%,max
date,8640,90,2025-01-01,96,NaN,NaN,NaN,NaN,NaN,NaN,NaN
city,8640,4,Bengaluru,2160,NaN,NaN,NaN,NaN,NaN,NaN,NaN
hour,8640.0,NaN,NaN,NaN,11.5,6.922587,0.0,5.75,11.5,17.25,23.0
day_of_week,8640.0,NaN,NaN,NaN,3.022222,1.999992,0.0,1.0,3.0,5.0,6.0
traffic_volume,8640.0,NaN,NaN,NaN,641.723495,350.431366,80.0,342.0,600.0,875.0,1840.0
average_speed,8640.0,NaN,NaN,NaN,54.424132,7.47489,26.1,49.4,55.1,60.0,68.0
traffic_status,8640,3,free_flow,5086,NaN,NaN,NaN,NaN,NaN,NaN,NaN
temperature,8640.0,NaN,NaN,NaN,24.222674,4.477924,8.6,21.2,24.5,27.4,37.5
rainfall,8640.0,NaN,NaN,NaN,0.338194,1.154155,0.0,0.0,0.0,0.0,18.3
relative_humidity,8640.0,NaN,NaN,NaN,56.897454,18.28553,20.0,42.9,52.5,69.2,100.0


## 4. Explore generated demand patterns

Every pattern in these charts comes from the synthetic generator. The charts help inspect the sample; they are not evidence about real cities.

In [4]:
hourly = df.groupby("hour", as_index=False)[TARGET_COLUMN].mean()
fig, ax = plt.subplots(figsize=(8, 4.5))
sns.lineplot(data=hourly, x="hour", y=TARGET_COLUMN, marker="o", color="#2b7864", ax=ax)
ax.set(xticks=range(0, 24, 2), title="Synthetic Training Demand by Hour",
       xlabel="Hour of day", ylabel="Mean generated sessions/hour")
fig.tight_layout()
plt.show()

fig, ax = plt.subplots(figsize=(8, 4.5))
sns.barplot(data=df, x="weather_condition", y=TARGET_COLUMN, errorbar=None,
            color="#83b9a5", ax=ax)
ax.set(title="Synthetic Training Demand by Generated Weather",
       xlabel="Weather category", ylabel="Mean generated sessions/hour")
fig.tight_layout()
plt.show()

sample = df.sample(min(1800, len(df)), random_state=42)
fig, ax = plt.subplots(figsize=(8, 4.5))
sns.regplot(data=sample, x="temperature", y=TARGET_COLUMN,
            scatter_kws={"alpha": 0.25, "s": 16},
            line_kws={"color": "#c25a36"}, ax=ax)
ax.set(title="Synthetic Temperature and Generated Demand",
       xlabel="Temperature (°C)", ylabel="Generated sessions/hour")
fig.tight_layout()
plt.show()

C:\Users\daksh\AppData\Local\Temp\ipykernel_20280\1436148162.py:7: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


C:\Users\daksh\AppData\Local\Temp\ipykernel_20280\1436148162.py:15: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


C:\Users\daksh\AppData\Local\Temp\ipykernel_20280\1436148162.py:25: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


### Traffic volume is used for EDA only

The synthetic traffic-volume field contributes to the generated target and can be plotted for exploration. It is not a model feature because the configured public sources do not provide an appropriate citywide volume count.

In [5]:
sample = df.sample(min(1800, len(df)), random_state=42)
fig, ax = plt.subplots(figsize=(8, 4.5))
sns.regplot(data=sample, x="traffic_volume", y=TARGET_COLUMN,
            scatter_kws={"alpha": 0.25, "s": 16},
            line_kws={"color": "#c25a36"}, ax=ax)
ax.set(title="Generated Traffic Volume and Generated Demand (EDA Only)",
       xlabel="Generated vehicles/hour", ylabel="Generated sessions/hour")
fig.tight_layout()
plt.show()

C:\Users\daksh\AppData\Local\Temp\ipykernel_20280\2947491411.py:9: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## 5. Prepare model features

The shared pipeline uses city, hour, weekday, traffic status, temperature, rainfall, weather category, and relative humidity. `peak_period` is derived from the hour. Twenty percent of synthetic traffic-status values are set to `unknown` in a reproducible augmentation so the pipeline can accept missing optional traffic data.

In [6]:
rows_before_cleaning = len(df)
training = df.drop_duplicates().dropna(subset=[TARGET_COLUMN]).copy()
training["hour"] = training["hour"].astype(int)
training["day_of_week"] = training["day_of_week"].astype(int)
training["traffic_status"] = training["average_speed"].map(traffic_status_from_speed)
unknown_mask = np.random.default_rng(42).random(len(training)) < 0.20
training.loc[unknown_mask, "traffic_status"] = "unknown"
model_data = add_engineered_features(training)
X = model_data[FEATURE_COLUMNS]
y = model_data[TARGET_COLUMN].astype(float)
print("Features:", FEATURE_COLUMNS)
print("Target:", TARGET_COLUMN)
print("Rows after cleaning:", len(model_data), "| unknown-traffic augmentation:", round(unknown_mask.mean(), 3))
display(model_data[["hour", "traffic_status", "peak_period", TARGET_COLUMN]].head())

Features: ['city', 'hour', 'day_of_week', 'traffic_status', 'temperature', 'rainfall', 'weather_condition', 'relative_humidity', 'peak_period']
Target: charging_demand
Rows after cleaning: 8640 | unknown-traffic augmentation: 0.203


,hour,traffic_status,peak_period,charging_demand
0,0,free_flow,off_peak,8
1,1,free_flow,off_peak,6
2,2,free_flow,off_peak,11
3,3,free_flow,off_peak,11
4,4,unknown,off_peak,8


## 6. Train and evaluate the model

A random 80/20 split and fixed seed make this classroom run repeatable. The metrics apply only to a held-out portion of the generated sample. They do not measure expected real-world accuracy.

In [7]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42
)
model = build_model()
model.fit(X_train, y_train)
predictions = model.predict(X_test)
mse = float(mean_squared_error(y_test, predictions))
metrics = {
    "mae": float(mean_absolute_error(y_test, predictions)),
    "mse": mse,
    "rmse": float(np.sqrt(mse)),
    "r2": float(r2_score(y_test, predictions)),
}
for name, value in metrics.items():
    print(f"{name.upper()}: {value:.3f}")
print(f"Train rows: {len(X_train):,} | held-out rows: {len(X_test):,}")

MAE: 3.474
MSE: 19.792
RMSE: 4.449
R2: 0.636
Train rows: 6,912 | held-out rows: 1,728


### Actual vs predicted values

In [8]:
fig, ax = plt.subplots(figsize=(6, 6))
ax.scatter(y_test, predictions, alpha=0.35, color="#2b7864", edgecolor="none")
upper = max(float(y_test.max()), float(np.max(predictions))) * 1.05
ax.plot([0, upper], [0, upper], "--", color="#c25a36")
ax.set(xlim=(0, upper), ylim=(0, upper),
       title="Synthetic Holdout: Actual vs Predicted",
       xlabel="Generated sessions/hour", ylabel="Predicted sessions/hour")
fig.tight_layout()
plt.show()

C:\Users\daksh\AppData\Local\Temp\ipykernel_20280\2806316106.py:9: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## 7. Inspect model importance and observations

The values below are calculated from the fitted Random Forest. Importance describes how the model used features in this synthetic fit; it is not a causal explanation.

In [9]:
importance = calculate_feature_importance(model)
display(pd.Series(importance, name="importance").to_frame())
insights = calculate_training_insights(model_data)
for observation in insights["observations"]:
    print("•", observation)

fig, ax = plt.subplots(figsize=(8, 4.5))
sns.barplot(x=list(importance.values()), y=list(importance.keys()), color="#2b7864", ax=ax)
ax.set(title="Calculated Random Forest Feature Importance",
       xlabel="Aggregated impurity importance", ylabel="Feature")
fig.tight_layout()
plt.show()

,importance
peak_period,0.597146
relative_humidity,0.098767
temperature,0.092707
hour,0.070751
day_of_week,0.048579
city,0.044242
traffic_status,0.024910
rainfall,0.011583
weather_condition,0.011314


• Generated examples have their highest hourly mean at 18:00 (23.7 sessions/hour) and lowest at 02:00 (6.9 sessions/hour).
• The generated weekend mean is 1.5 sessions/hour lower than the weekday mean.
• Clear has the highest generated mean among weather categories (14.2 sessions/hour).
• Generated traffic volume and target have Pearson r=0.78; volume is an EDA-only field, not a prediction input.


C:\Users\daksh\AppData\Local\Temp\ipykernel_20280\3360237244.py:12: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## 8. Save the same project artifacts

This saves the model pipeline and calculated metadata in the paths used by the Flask app. The training script is the preferred repeatable command (`python train_model.py`); this cell shows the same outputs from the notebook run.

In [10]:
metrics.update({
    "rows_before_cleaning": int(rows_before_cleaning),
    "rows_after_cleaning": int(len(model_data)),
    "train_rows": int(len(X_train)),
    "test_rows": int(len(X_test)),
    "target": TARGET_COLUMN,
    "target_unit": "model-estimated charging sessions per city-hour",
    "dataset_type": "synthetic",
    "missing_traffic_augmentation_share": 0.20,
    "random_state": 42,
    "features": FEATURE_COLUMNS,
    "categorical_features": CATEGORICAL_COLUMNS,
    "numeric_features": NUMERIC_COLUMNS,
    "model": "RandomForestRegressor",
})
MODEL_PATH.parent.mkdir(parents=True, exist_ok=True)
dump(model, MODEL_PATH)
METRICS_PATH.write_text(json.dumps(metrics, indent=2), encoding="utf-8")
IMPORTANCE_PATH.write_text(json.dumps(importance, indent=2), encoding="utf-8")
TRAINING_INSIGHTS_PATH.write_text(json.dumps(insights, indent=2), encoding="utf-8")
save_visualizations(model_data, y_test, predictions, importance)
print("Saved model, metrics, feature importance, training observations, and charts.")

Saved model, metrics, feature importance, training observations, and charts.


## 9. Optional public API examples

These examples call the same cached clients as the web app. They are wrapped so internet failure, rate limits, or missing credentials produce a clear note rather than stopping the notebook. Historical weather from Open-Meteo is reanalysis. Historical traffic is not configured, and no synthetic substitute is used.

In [11]:
from datetime import date, timedelta
from data_sources import DataSourceError, get_current_weather, get_historical_weather, get_traffic_data, get_weather_forecast

city = "Bengaluru"
try:
    current = get_current_weather(city)
    print("Current weather source:", current["_meta"]["source"])
    print({key: current.get(key) for key in ["weather_time", "temperature", "rainfall", "weather_condition", "relative_humidity"]})
except DataSourceError as exc:
    print("Current weather unavailable:", exc)

traffic = get_traffic_data(city)
print("Traffic status:", traffic["traffic_status"], "| source:", traffic["source"])

yesterday = date.today() - timedelta(days=1)
try:
    archived = get_historical_weather(city, yesterday.isoformat(), yesterday.isoformat())
    print("Historical weather rows:", len(archived["records"]), "| source:", archived["_meta"]["source"])
except DataSourceError as exc:
    print("Historical weather unavailable:", exc)

tomorrow = date.today() + timedelta(days=1)
try:
    forecast = get_weather_forecast(city, tomorrow.isoformat(), 18)
    print("Future weather type:", forecast["data_kind"], "| source:", forecast["source"])
except DataSourceError as exc:
    print("Future weather unavailable:", exc)

Current weather source: Open-Meteo current weather
{'weather_time': '2026-10-06T20:45', 'temperature': 21.8, 'rainfall': 0.0, 'weather_condition': 'Clear', 'relative_humidity': 84.0}
Traffic status: unknown | source: Unavailable: optional TomTom key is not configured


Historical weather rows: 24 | source: Open-Meteo historical weather reanalysis


Future weather type: forecast | source: Open-Meteo hourly forecast (up to 16 days)


## Conclusion

The notebook demonstrates a complete synthetic-data regression workflow and shows how the app retrieves current, historical, and forecast weather while labeling unavailable traffic and charging data. Predictions are educational model estimates, not observed charging-session counts.